# Does a Cell Segmentation Model Survive a Change of Cell Type?
### Fine-tuning depth vs. cross-domain robustness on the Sartorius cell dataset

**Author:** Wajid Ali Khan · [GitHub](https://github.com/wajidali99) · [Hugging Face](https://huggingface.co/Wajiddev99)

**Question.** A segmentation model trained on one kind of cell image often fails on another (new cell line, microscope or batch).
Does *how much* of a pretrained encoder we fine-tune change this?

**Setup.**
- Data: Kaggle *Sartorius – Cell Instance Segmentation* (phase-contrast neuronal cells; 3 cell types: `shsy5y`, `astro`, `cort`).
- Model: U-Net with an ImageNet-pretrained ResNet-34 encoder.
- Three fine-tuning regimes:
  1. **Frozen** – encoder frozen, only decoder trained (cheap, like linear-probe / PEFT-style)
  2. **Partial** – last encoder stage (`layer4`) + decoder trained
  3. **Full** – everything trained
- For each regime, train on **one** cell type and test on **all three** → a 3×3 transfer matrix (Dice, IoU).

**Why.** Off-diagonal cells of the matrix measure domain shift. Comparing matrices across regimes shows whether more adaptation helps in-domain but hurts (or helps) out-of-domain.

> Add the dataset: *Add Data → Competitions → "sartorius-cell-instance-segmentation"*. Turn on **GPU (T4 or P100)**.

In [ ]:
!pip -q install segmentation-models-pytorch==0.3.4 albumentations==1.4.18

In [ ]:
import os, random, time, json, copy
import numpy as np, pandas as pd
import cv2, torch, torch.nn as nn
import matplotlib.pyplot as plt, seaborn as sns
import albumentations as A
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

CFG = dict(
    root="/kaggle/input/sartorius-cell-instance-segmentation",
    img_h=512, img_w=704,          # 520x704 -> 512x704 (divisible by 32)
    epochs=20, batch_size=8, lr=3e-4, test_frac=0.2,
    encoder="resnet34", seed=42,
    regimes=["frozen", "partial", "full"],
    cell_types=["shsy5y", "astro", "cort"],
)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG["seed"]); print(DEVICE)

## 1. Data: decode instance RLEs into one binary foreground mask per image

In [ ]:
df = pd.read_csv(f"{CFG['root']}/train.csv")
print(df.shape); display(df.head(3))
print(df.groupby("cell_type")["id"].nunique().rename("n_images"))

def rle_decode(rle, h, w):
    s = np.asarray(rle.split(), dtype=int)
    starts, lengths = s[0::2] - 1, s[1::2]
    m = np.zeros(h * w, dtype=np.uint8)
    for a, l in zip(starts, lengths):
        m[a:a + l] = 1
    return m.reshape(h, w)

# one row per image
img_df = df.groupby("id").agg(cell_type=("cell_type", "first"), h=("height", "first"),
                              w=("width", "first"), annotations=("annotation", list)).reset_index()
img_df["n_cells"] = img_df["annotations"].str.len()
display(img_df.groupby("cell_type")["n_cells"].describe())

In [ ]:
# per cell type 80/20 split by image (same split reused for every experiment)
rng = np.random.RandomState(CFG["seed"])
img_df["split"] = "train"
for ct in CFG["cell_types"]:
    ids = img_df.index[img_df.cell_type == ct].values.copy(); rng.shuffle(ids)
    img_df.loc[ids[: int(len(ids) * CFG["test_frac"])], "split"] = "test"
print(pd.crosstab(img_df.cell_type, img_df.split))

In [ ]:
# cache images + masks in RAM (606 images, small)
CACHE = {}
for r in tqdm(img_df.itertuples(), total=len(img_df)):
    img = cv2.imread(f"{CFG['root']}/train/{r.id}.png", cv2.IMREAD_GRAYSCALE)
    mask = np.zeros((r.h, r.w), np.uint8)
    for a in r.annotations:
        mask |= rle_decode(a, r.h, r.w)
    img = cv2.resize(img, (CFG["img_w"], CFG["img_h"]), interpolation=cv2.INTER_LINEAR)
    mask = cv2.resize(mask, (CFG["img_w"], CFG["img_h"]), interpolation=cv2.INTER_NEAREST)
    CACHE[r.id] = (img, mask)

fig, ax = plt.subplots(2, 3, figsize=(15, 7))
for j, ct in enumerate(CFG["cell_types"]):
    i = img_df[img_df.cell_type == ct].id.iloc[0]
    ax[0, j].imshow(CACHE[i][0], cmap="gray"); ax[0, j].set_title(ct)
    ax[1, j].imshow(CACHE[i][1]); ax[1, j].set_title(f"{ct} – mask")
[a.axis("off") for a in ax.ravel()]; plt.tight_layout(); plt.show()

**Intensity statistics per cell type.** If the three domains look different at pixel level, that already hints at domain shift.

In [ ]:
stats = []
for r in img_df.itertuples():
    im, m = CACHE[r.id]
    stats.append(dict(cell_type=r.cell_type, mean=im.mean(), std=im.std(), fg_frac=m.mean()))
stats = pd.DataFrame(stats)
display(stats.groupby("cell_type").agg(["mean", "std"]).round(3))

## 2. Dataset, augmentation, model

In [ ]:
train_tf = A.Compose([A.HorizontalFlip(), A.VerticalFlip(),
                      A.RandomBrightnessContrast(0.15, 0.15, p=0.5),
                      A.GaussNoise(var_limit=(5, 20), p=0.2)])

class CellDS(Dataset):
    def __init__(self, ids, tf=None): self.ids, self.tf = list(ids), tf
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        img, m = CACHE[self.ids[i]]
        if self.tf: o = self.tf(image=img, mask=m); img, m = o["image"], o["mask"]
        img = img.astype(np.float32) / 255.0
        img = (img - img.mean()) / (img.std() + 1e-6)          # per-image normalisation
        img = np.repeat(img[None], 3, 0)                        # 3 channels for ImageNet encoder
        return torch.from_numpy(img), torch.from_numpy(m[None].astype(np.float32))

def ids_of(ct, split): return img_df[(img_df.cell_type == ct) & (img_df.split == split)].id.values

def build_model(regime):
    model = smp.Unet(CFG["encoder"], encoder_weights="imagenet", in_channels=3, classes=1)
    for p in model.encoder.parameters(): p.requires_grad = regime == "full"
    if regime == "partial":
        for p in model.encoder.layer4.parameters(): p.requires_grad = True
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    return model.to(DEVICE), n_tr, n_all

for rg in CFG["regimes"]:
    _, a, b = build_model(rg); print(f"{rg:8s} trainable {a/1e6:5.2f}M / {b/1e6:.2f}M ({100*a/b:.0f}%)")

## 3. Training and evaluation

In [ ]:
dice_loss = smp.losses.DiceLoss("binary"); bce = nn.BCEWithLogitsLoss()
def loss_fn(lo, y): return bce(lo, y) + dice_loss(lo, y)

@torch.no_grad()
def evaluate(model, ids):
    model.eval(); dices, ious = [], []
    for x, y in DataLoader(CellDS(ids), batch_size=8, num_workers=2):
        p = (torch.sigmoid(model(x.to(DEVICE))) > 0.5).float(); y = y.to(DEVICE)
        inter = (p * y).sum((1, 2, 3)); ps, ys = p.sum((1, 2, 3)), y.sum((1, 2, 3))
        dices += ((2 * inter + 1) / (ps + ys + 1)).tolist()
        ious  += ((inter + 1) / (ps + ys - inter + 1)).tolist()
    return float(np.mean(dices)), float(np.mean(ious))

def train_one(source, regime):
    seed_all(CFG["seed"])
    model, n_tr, _ = build_model(regime)
    dl = DataLoader(CellDS(ids_of(source, "train"), train_tf), batch_size=CFG["batch_size"],
                    shuffle=True, num_workers=2, drop_last=True)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=CFG["lr"], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, CFG["epochs"] * len(dl))
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == "cuda")
    hist = []
    for ep in range(CFG["epochs"]):
        model.train(); tot = 0
        if regime != "full":                     # keep frozen BatchNorm statistics fixed
            model.encoder.eval()
            if regime == "partial": model.encoder.layer4.train()
        for x, y in dl:
            x, y = x.to(DEVICE), y.to(DEVICE); opt.zero_grad()
            with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
                l = loss_fn(model(x), y)
            scaler.scale(l).backward(); scaler.step(opt); scaler.update(); sched.step(); tot += l.item()
        hist.append(tot / len(dl))
    return model, hist, n_tr

In [ ]:
results, histories, models = [], {}, {}
for regime in CFG["regimes"]:
    for src in CFG["cell_types"]:
        t0 = time.time()
        model, hist, n_tr = train_one(src, regime)
        histories[(regime, src)] = hist
        for tgt in CFG["cell_types"]:
            d, iou = evaluate(model, ids_of(tgt, "test"))
            results.append(dict(regime=regime, source=src, target=tgt, dice=d, iou=iou,
                                in_domain=src == tgt, trainable_M=n_tr / 1e6))
        models[(regime, src)] = model.cpu()
        print(f"{regime:8s} src={src:7s} done in {time.time()-t0:.0f}s | "
              + " ".join(f"{r['target']}={r['dice']:.3f}" for r in results[-3:]))

res = pd.DataFrame(results); res.to_csv("results.csv", index=False); display(res.round(3))

## 4. Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, rg in zip(axes, CFG["regimes"]):
    mat = res[res.regime == rg].pivot(index="source", columns="target", values="dice") \
            .loc[CFG["cell_types"], CFG["cell_types"]]
    sns.heatmap(mat, annot=True, fmt=".3f", vmin=0.3, vmax=0.9, cmap="viridis", ax=ax, cbar=False)
    ax.set_title(f"{rg} fine-tuning – Dice"); ax.set_xlabel("test cell type"); ax.set_ylabel("train cell type")
plt.tight_layout(); plt.savefig("transfer_matrices.png", dpi=150); plt.show()

In [ ]:
# In-domain vs cross-domain summary; "drop" = how much Dice is lost when the cell type changes
summ = res.groupby(["regime", "in_domain"])["dice"].mean().unstack()
summ.columns = ["cross_domain_dice", "in_domain_dice"]
summ["drop"] = summ["in_domain_dice"] - summ["cross_domain_dice"]
summ["trainable_M"] = res.groupby("regime")["trainable_M"].first()
summ = summ.loc[CFG["regimes"]]; display(summ.round(3))

ax = summ[["in_domain_dice", "cross_domain_dice"]].plot.bar(figsize=(7, 4), rot=0)
ax.set_ylabel("mean Dice"); ax.set_title("In-domain vs cross-domain Dice by fine-tuning regime")
plt.tight_layout(); plt.savefig("in_vs_cross.png", dpi=150); plt.show()

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8), sharey=True)
for ax, src in zip(axes, CFG["cell_types"]):
    for rg in CFG["regimes"]: ax.plot(histories[(rg, src)], label=rg)
    ax.set_title(f"train on {src}"); ax.set_xlabel("epoch")
axes[0].set_ylabel("BCE + Dice loss"); axes[0].legend(); plt.tight_layout(); plt.show()

## 5. Where does it fail? Qualitative cross-domain examples

In [ ]:
@torch.no_grad()
def show_examples(regime, src):
    m = models[(regime, src)].to(DEVICE).eval()
    fig, ax = plt.subplots(3, 3, figsize=(14, 11))
    for j, tgt in enumerate(CFG["cell_types"]):
        x, y = CellDS(ids_of(tgt, "test")[:1])[0]
        p = (torch.sigmoid(m(x[None].to(DEVICE)))[0, 0] > 0.5).cpu().numpy()
        ax[0, j].imshow(x[0], cmap="gray"); ax[0, j].set_title(f"test: {tgt}")
        ax[1, j].imshow(y[0]); ax[1, j].set_title("ground truth")
        err = np.zeros((*p.shape, 3)); g = y[0].numpy() > 0
        err[p & g] = [0, 1, 0]; err[p & ~g] = [1, 0, 0]; err[~p & g] = [0, 0.4, 1]
        ax[2, j].imshow(err); ax[2, j].set_title("green=TP  red=FP  blue=FN")
    [a.axis("off") for a in ax.ravel()]
    fig.suptitle(f"{regime} model trained on {src}"); plt.tight_layout(); plt.show(); m.cpu()

show_examples("full", "shsy5y")
show_examples("frozen", "shsy5y")

## 6. Findings  *(write these after running — use your own numbers)*

1. **In-domain:** which regime gives the best Dice on the same cell type? By how much over *frozen*?
2. **Cross-domain:** which train→test pair is hardest? Does it match the intensity/foreground stats from section 1?
3. **Trade-off:** does *full* fine-tuning increase the in-domain vs cross-domain gap (overfitting to one domain), or close it?
4. **Error type:** from section 5, are cross-domain errors mostly false positives (debris/background) or false negatives (missed thin processes, e.g. astrocytes)?

## 7. Limitations & next steps
- Semantic (foreground) masks only; instance separation (e.g. watershed or Cellpose-style flows) not evaluated.
- One seed; repeat with 3 seeds for error bars.
- Next: (a) add LoRA adapters on a foundation encoder (e.g. SAM / a ViT) as a true PEFT arm; (b) test-time adaptation (BatchNorm statistics update on target images); (c) train on two cell types, test on the held-out third.

In [ ]:
# Optional extra: test-time BatchNorm adaptation (no labels) — a cheap domain-adaptation baseline
@torch.no_grad()
def tta_bn(model, ids, passes=2):
    m = copy.deepcopy(model).to(DEVICE)
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d): mod.reset_running_stats(); mod.momentum = None
    m.train()
    for _ in range(passes):
        for x, _ in DataLoader(CellDS(ids), batch_size=8): m(x.to(DEVICE))
    return m

rows = []
for (rg, src), mdl in models.items():
    for tgt in CFG["cell_types"]:
        if tgt == src: continue
        adapted = tta_bn(mdl, ids_of(tgt, "train"))   # unlabeled target images
        d, _ = evaluate(adapted, ids_of(tgt, "test"))
        base = res[(res.regime == rg) & (res.source == src) & (res.target == tgt)].dice.item()
        rows.append(dict(regime=rg, source=src, target=tgt, dice_before=base, dice_after_bn=d))
tta = pd.DataFrame(rows); tta["gain"] = tta.dice_after_bn - tta.dice_before
display(tta.groupby("regime")[["dice_before", "dice_after_bn", "gain"]].mean().round(3))
tta.to_csv("tta_bn_results.csv", index=False)